# TradeSense · Component 03 · notebook 42: multi-horizon view (15 minutes, 1 hour, 1 day) built from the 1-minute scores (CPU, no training)

**Why.** Manipulation and abnormal behaviour unfold over time, so a user should be able to ask "what happened in the last 15 minutes / hour / day?", not only "was this one minute odd?". The detectors stay 1-minute models; this notebook **rolls their fused scores up** into tumbling windows of 15, 60 and 1,440 bars and checks, with the same pre-registered discipline, what the roll-up adds.

**Window score (declared before the run).** `window score = maximum fusion v0 score inside the window` (secondary, reported only: number of Watch-8 bars). It is turned into a **0-100 "unusualness" value** with a fixed empirical distribution of the same horizon built from the *genuine DEV out-of-fold* windows: `score100 = percentage of normal windows with a lower peak`. So "97" means "more unusual than 97% of normal 15-minute / hourly / daily windows".

**Tiers per horizon (share of normal windows flagged, fixed from genuine DEV windows):**

| horizon | Watch (amber) | Alert (red) |
|---|---|---|
| 15 minutes | 5% | 1% |
| 1 hour | 8% | 2% |
| 1 day | 25% | 10% |

**Pre-registered hypotheses (CAL-3, fusion v0).**
- **H-M1 (no events lost):** at 15 minutes and at 1 hour, the macro-8 event recall of the Alert tier (an event counts as found if ANY window overlapping it is flagged) is >= the 1-minute Alert recall recomputed here (0.778 in notebook 35) minus 0.02, and the genuine flagged share stays within 0.5x-2x of the target.
- **H-M2 (aggregation helps the ambiguous family):** abnormal_volume_burst recall at the 1-hour Watch tier exceeds its 1-minute Alert recall by >= 0.10 (paired bootstrap CI above 0). Exploratory: it is reported either way.
- **H-M3 (case, not a detection test):** the real 5 August 2024 UTC day gets a daily score100 >= 90 for BTCUSDT and ETHUSDT. This is one real day and says nothing about manipulation, only whether the daily view reacts to a real market shock.
- **Stated limit:** CAL-3 injects about 6-7 events per day, far more than real life, so *every* injected day is positive; the 1-day horizon therefore cannot be scored as detection (it is reported as a distribution and as the 5 August case). 15-minute and 1-hour windows are scored as detection, where about a quarter / a third of windows contain an event.

TEST-3, TEST-B and every TGNN test day are untouched. Nothing here changes a frozen rule; the minute-level Alert is recomputed and must equal notebook 35 and 40.

In [ ]:
import os, sys, json, time, hashlib, warnings, datetime
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not in Colab -> using TRADESENSE_BASE.')
BASE = Path(os.environ.get('TRADESENSE_BASE', '/content/drive/MyDrive/research datasets'))
L3 = BASE/'layer3_v3'; OUT = BASE/'models_b2_v1'
SYMBOLS = ['BTCUSDT', 'ETHUSDT']
MOCK = os.environ.get('TS_MOCK') == '1'; NBOOT = 200 if MOCK else 2000
LINES = []
def say(s=''): print(s); LINES.append(str(s))
def sha_file(p, n=16): return hashlib.sha256(Path(p).read_bytes()).hexdigest()[:n]
hf = L3/'protocol_bar_models_v2.sha256'; pf = L3/'protocol_bar_models_v2.md'
assert hf.exists() and pf.exists(), 'run notebook 27 first'
ph = hashlib.sha256(pf.read_text().encode()).hexdigest(); assert ph == hf.read_text().split()[0], 'frozen protocol B2 was changed'
for f in ['declared_fusion_v0.json', 'fusion_v0_cal3_table.csv', 'declared_watch_v1.json']:
    assert (OUT/f).exists(), f'{f} missing: run notebooks 35 (patched) and 40 to the end first'
for s in SYMBOLS: assert (OUT/'scores'/f'fusion_v0_{s}.npz').exists(), 'run notebook 35 first'
sys.path.insert(0, str(OUT)); import importlib; import b2_lib as B; importlib.reload(B)
say('protocol B2 intact, sha256 ' + ph[:16] + '; library sha256 ' + sha_file(OUT/'b2_lib.py') + ('  [MOCK RUN]' if MOCK else ''))
HOR = {'15 min': 15, '1 hour': 60, '1 day': 1440}
FRAC = {'15 min': {'Watch': 0.05, 'Alert': 0.01}, '1 hour': {'Watch': 0.08, 'Alert': 0.02}, '1 day': {'Watch': 0.25, 'Alert': 0.10}}

Mounted at /content/drive
protocol B2 intact, sha256 443f99d5faf5929f; library sha256 1b0b46630763c944


## 1 · Data and window scores

In [ ]:
AOLEN = 31 * 1440; CAL_LO = 1440; DEV_LO, DEV_HI = 90720, 129600
CALV = B.valid_mask(AOLEN, CAL_LO, AOLEN)
FS = {}; EV = {}
for s in SYMBOLS:
    z = np.load(OUT/'scores'/f'fusion_v0_{s}.npz'); assert not any('TEST' in k for k in z.files)
    FS[s] = {k: z[k] for k in ['GEN_AO', 'CAL_r0', 'OOF_GEN_L1']}
    e = pd.read_csv(L3/f'l3v3_{s}_CAL_events.csv'); e = e[e.replica == 0].reset_index(drop=True); assert e.t0.min() >= CAL_LO and (e.t0 + e.dur).max() <= AOLEN; EV[s] = e
FAMN = B.FAMS; FAMS = list(FAMN); EALL = pd.concat([EV[s].assign(sym=s) for s in SYMBOLS], ignore_index=True)
DW = json.load(open(OUT/'declared_watch_v1.json')); QA = DW['alert']['q']; QW8 = DW['watch']['q']['8']
def thr_roll(g, q): t = B.rolling_thr(g, q); return np.where(np.isnan(t), np.inf, t)
def event_hits(a, ev):
    c = np.concatenate([[0], np.cumsum(np.nan_to_num(a).astype(np.int64))]); t0 = ev.t0.to_numpy(); du = ev.dur.to_numpy(); return (c[t0 + du] - c[t0]) > 0
def macro8_of(h):
    d = pd.DataFrame({'sym': EALL.sym, 'fam': EALL.family, 'h': h}); return float(d.groupby(['sym', 'fam']).h.mean().groupby('fam').mean().mean())
# minute-level references recomputed from the declared quantiles (notebook 40)
MIN_ALERT = {}; MIN_W8 = {}; W8BARS = {}
for s in SYMBOLS:
    thr = thr_roll(FS[s]['GEN_AO'], QA[s]); MIN_ALERT[s] = event_hits(FS[s]['CAL_r0'] > thr, EV[s])
    thr8 = thr_roll(FS[s]['GEN_AO'], QW8[s]); MIN_W8[s] = event_hits(FS[s]['CAL_r0'] > thr8, EV[s]); W8BARS[s] = {'GEN_AO': (FS[s]['GEN_AO'] > thr8).astype(float), 'CAL_r0': (FS[s]['CAL_r0'] > thr8).astype(float)}
H_MIN = np.concatenate([MIN_ALERT[s] for s in SYMBOLS])
def win_max(g, lo, n, H):
    out = np.full(n, np.nan)
    for k in range(n):
        w = g[lo + k * H: lo + (k + 1) * H]
        if np.isfinite(w).mean() >= 0.5: out[k] = np.nanmax(w)
    return out
NW_CAL = {h: (AOLEN - CAL_LO) // H for h, H in HOR.items()}; NW_DEV = {h: (DEV_HI - (DEV_LO + 1440)) // H for h, H in HOR.items()}
WS = {}                                                     # window scores
for s in SYMBOLS:
    for h, H in HOR.items():
        WS[(s, h, 'dev')] = win_max(FS[s]['OOF_GEN_L1'], DEV_LO + 1440, NW_DEV[h], H)
        WS[(s, h, 'gen')] = win_max(FS[s]['GEN_AO'], CAL_LO, NW_CAL[h], H)
        WS[(s, h, 'inj')] = win_max(FS[s]['CAL_r0'], CAL_LO, NW_CAL[h], H)
THR = {}; ECDF = {}
for s in SYMBOLS:
    for h in HOR:
        d = WS[(s, h, 'dev')]; d = d[np.isfinite(d)]; ECDF[(s, h)] = np.percentile(d, np.arange(0, 101))
        for tier, fr in FRAC[h].items(): THR[(s, h, tier)] = float(np.quantile(d, 1 - fr))
def score100(s, h, x): return float(np.interp(x, ECDF[(s, h)], np.arange(0, 101))) if np.isfinite(x) else np.nan
say('windows: ' + str({h: f'{NW_DEV[h]} genuine DEV, {NW_CAL[h]} CAL per symbol' for h in HOR}))

windows: {'15 min': '2496 genuine DEV, 2880 CAL per symbol', '1 hour': '624 genuine DEV, 720 CAL per symbol', '1 day': '26 genuine DEV, 30 CAL per symbol'}


## 2 · Unit tests

In [ ]:
ref = float(pd.read_csv(OUT/'fusion_v0_cal3_table.csv').query("method == 'fusion' and budget == 2")['macro 8'].iloc[0])
assert abs(macro8_of(H_MIN) - ref) < 5e-4, (macro8_of(H_MIN), ref); say(f'PASS T01 the 1-minute Alert recomputed here equals notebooks 35 and 40 (macro 8 {ref:.3f})')
g = np.arange(100.0); w = win_max(g, 10, 4, 20); assert list(w) == [29, 49, 69, 89]; say('PASS T02 window maximum equals a hand calculation')
ev_ = pd.DataFrame({'t0': [5, 35], 'dur': [20, 3]}); H_ = 15; lo_ = 0; rng = [(( a) // H_, (a + d - 1) // H_) for a, d in zip(ev_.t0, ev_.dur)]
plain = [[k for k in range(10) if (k * H_ < a + d) and ((k + 1) * H_ > a)] for a, d in zip(ev_.t0, ev_.dur)]
assert all(list(range(r[0], r[1] + 1)) == p for r, p in zip(rng, plain)); say('PASS T03 window-overlap rule equals a plain loop')
for s in SYMBOLS:
    for h in HOR: assert THR[(s, h, 'Watch')] <= THR[(s, h, 'Alert')] and (np.diff(ECDF[(s, h)]) >= 0).all()
for h in HOR: assert 0 <= score100('BTCUSDT', h, 1e9) <= 100 and 0 <= score100('BTCUSDT', h, -1e9) <= 100
say('PASS T04 Watch threshold <= Alert threshold; score100 is monotone and inside 0-100')
say('PASS T05 only CAL-3 and DEV out-of-fold series are read (no TEST key in the score files)')

PASS T01 the 1-minute Alert recomputed here equals notebooks 35 and 40 (macro 8 0.778)
PASS T02 window maximum equals a hand calculation
PASS T03 window-overlap rule equals a plain loop
PASS T04 Watch threshold <= Alert threshold; score100 is monotone and inside 0-100
PASS T05 only CAL-3 and DEV out-of-fold series are read (no TEST key in the score files)


## 3 · Window-level detection (15 minutes, 1 hour)

In [ ]:
from sklearn.metrics import roc_auc_score
def overlap_windows(ev, H):
    a = ((ev.t0.to_numpy() - CAL_LO) // H).astype(int); b = ((ev.t0.to_numpy() + ev.dur.to_numpy() - 1 - CAL_LO) // H).astype(int); return a, b
def truth_mask(ev, H, n):
    m = np.zeros(n, bool); a, b = overlap_windows(ev, H)
    for x, y in zip(a, b): m[x:min(y, n - 1) + 1] = True
    return m
HITS = {}; ROWS = []; AUCS = []
for h, H in HOR.items():
    if h == '1 day': continue
    for tier in ['Watch', 'Alert']:
        hs = []; fl = []
        for s in SYMBOLS:
            n = NW_CAL[h]; inj = WS[(s, h, 'inj')]; gen = WS[(s, h, 'gen')]; flag = np.nan_to_num(inj, nan=-np.inf) > THR[(s, h, tier)]
            a, b = overlap_windows(EV[s], H); hs.append(np.array([flag[x:min(y, n - 1) + 1].any() for x, y in zip(a, b)])); fl.append(float((np.nan_to_num(gen, nan=-np.inf) > THR[(s, h, tier)]).mean()))
        HITS[(h, tier)] = np.concatenate(hs); gf = float(np.mean(fl)); wpd = 1440 / H
        ROWS.append({'horizon': h, 'tier': tier, 'target genuine share': FRAC[h][tier], 'CAL genuine share flagged': gf, 'genuine flagged windows/day': gf * wpd, 'macro 8 recall': macro8_of(HITS[(h, tier)])})
    for s in SYMBOLS:
        pos = truth_mask(EV[s], H, NW_CAL[h]); inj = WS[(s, h, 'inj')]; gen = WS[(s, h, 'gen')]; ok = np.isfinite(inj) & pos; gn = gen[np.isfinite(gen)]
        y = np.r_[np.ones(ok.sum()), np.zeros(len(gn))]; sc = np.r_[inj[ok], gn]; AUCS.append({'horizon': h, 'sym': s, 'positive windows': int(ok.sum()), 'share of injected windows positive': float(pos.mean()), 'window ROC-AUC (peak score)': float(roc_auc_score(y, sc))})
RT = pd.DataFrame(ROWS); AT = pd.DataFrame(AUCS).groupby('horizon').mean(numeric_only=True).reset_index()
say(''); say('--- detection at window level (an event is found if ANY window overlapping it is flagged) versus the 1-minute Alert (macro 8 %.3f) ---' % macro8_of(H_MIN)); say(RT.round(3).to_string(index=False))
say(''); say('window ROC-AUC (injected windows with an event versus genuine windows, peak score)'); say(AT.round(3).to_string(index=False))
FT = pd.DataFrame({'family': [FAMN[f] for f in FAMS], '1-min Alert': [float(H_MIN[(EALL.family == f).to_numpy()].mean()) for f in FAMS], '1-min Watch-8': [float(np.concatenate([MIN_W8[s] for s in SYMBOLS])[(EALL.family == f).to_numpy()].mean()) for f in FAMS], '15 min Alert': [float(HITS[('15 min', 'Alert')][(EALL.family == f).to_numpy()].mean()) for f in FAMS], '1 hour Watch': [float(HITS[('1 hour', 'Watch')][(EALL.family == f).to_numpy()].mean()) for f in FAMS], '1 hour Alert': [float(HITS[('1 hour', 'Alert')][(EALL.family == f).to_numpy()].mean()) for f in FAMS]})
say(''); say('--- recall per family (CAL-3) ---'); say(FT.round(2).to_string(index=False))
HM1 = bool(all(RT[(RT.horizon == h) & (RT.tier == 'Alert')]['macro 8 recall'].iloc[0] >= macro8_of(H_MIN) - 0.02 and 0.5 <= RT[(RT.horizon == h) & (RT.tier == 'Alert')]['CAL genuine share flagged'].iloc[0] / FRAC[h]['Alert'] <= 2 for h in ['15 min', '1 hour']))
vb = np.flatnonzero((EALL.family == 3).to_numpy()); rg = np.random.default_rng(0); dr = rg.integers(0, len(vb), size=(NBOOT, len(vb))); dif = (HITS[('1 hour', 'Watch')][vb][dr].mean(axis=1) - H_MIN[vb][dr].mean(axis=1)); gain = float(HITS[('1 hour', 'Watch')][vb].mean() - H_MIN[vb].mean()); lo = float(np.percentile(dif, 2.5)); HM2 = bool(gain >= 0.10 and lo > 0)
say(''); say(f'H-M1 (15 min and 1 hour Alert lose no events and keep their genuine share within 0.5x-2x): {HM1}')
say(f'H-M2 (volume burst, 1-hour Watch versus 1-minute Alert): gain {gain:+.2f} (95% CI {np.percentile(dif, 2.5):+.2f}..{np.percentile(dif, 97.5):+.2f}) -> {HM2}')


--- detection at window level (an event is found if ANY window overlapping it is flagged) versus the 1-minute Alert (macro 8 0.778) ---
horizon  tier  target genuine share  CAL genuine share flagged  genuine flagged windows/day  macro 8 recall
 15 min Watch                  0.05                      0.041                        3.950           0.828
 15 min Alert                  0.01                      0.009                        0.833           0.708
 1 hour Watch                  0.08                      0.062                        1.500           0.772
 1 hour Alert                  0.02                      0.015                        0.350           0.678

window ROC-AUC (injected windows with an event versus genuine windows, peak score)
horizon  positive windows  share of injected windows positive  window ROC-AUC (peak score)
 1 hour             283.5                               0.394                        0.887
 15 min             563.0                               0

## 4 · The 1-day view and the 5 August 2024 case

In [ ]:
say(''); say('--- 1-day horizon (reported as a distribution, not as detection: every injected day contains events) ---')
rows = []
for s in SYMBOLS:
    g = WS[(s, '1 day', 'gen')]; i = WS[(s, '1 day', 'inj')]
    rows.append({'sym': s, 'genuine days': int(np.isfinite(g).sum()), 'genuine median score100': float(np.nanmedian([score100(s, '1 day', x) for x in g])), 'injected days median score100': float(np.nanmedian([score100(s, '1 day', x) for x in i])), 'injected days with score100 >= 90': float(np.nanmean([score100(s, '1 day', x) >= 90 for x in i]))})
say(pd.DataFrame(rows).round(2).to_string(index=False))
# real case: 5 August 2024 UTC (bar 0 of the August series is 1 August 00:00 UTC, so 5 August is window index 3 after bar 1440)
K = 3; HM3 = True; case = []
for s in SYMBOLS:
    d1 = score100(s, '1 day', WS[(s, '1 day', 'gen')][K]); h1 = WS[(s, '1 hour', 'gen')][K * 24:(K + 1) * 24]; m15 = WS[(s, '15 min', 'gen')][K * 96:(K + 1) * 96]
    top_h = int(np.nanargmax(h1)); top_m = int(np.nanargmax(m15)); case.append({'sym': s, '1-day score100 (5 Aug UTC)': d1, 'peak hour (UTC)': f'{top_h:02d}:00', 'peak-hour score100': score100(s, '1 hour', h1[top_h]), 'peak 15-min window (UTC)': f'{top_m // 4:02d}:{(top_m % 4) * 15:02d}', 'peak 15-min score100': score100(s, '15 min', m15[top_m])}); HM3 &= bool(d1 >= 90)
say(''); say('real case: 5 August 2024 (genuine series, no injection)'); say(pd.DataFrame(case).round(1).to_string(index=False)); say(f'H-M3 (daily score100 of 5 August >= 90 for both symbols): {HM3}')


--- 1-day horizon (reported as a distribution, not as detection: every injected day contains events) ---
    sym  genuine days  genuine median score100  injected days median score100  injected days with score100 >= 90
BTCUSDT            30                    45.00                          100.0                               1.00
ETHUSDT            30                    60.51                          100.0                               0.97

real case: 5 August 2024 (genuine series, no injection)
    sym  1-day score100 (5 Aug UTC) peak hour (UTC)  peak-hour score100 peak 15-min window (UTC)  peak 15-min score100
BTCUSDT                        73.3           04:00                98.5                    04:15                  99.2
ETHUSDT                        64.4           23:00                97.3                    23:00                  99.1
H-M3 (daily score100 of 5 August >= 90 for both symbols): False


## 5 · Save and declare

In [ ]:
RT.to_csv(OUT/'multihorizon_cal3_table.csv', index=False); FT.to_csv(OUT/'multihorizon_cal3_by_family.csv', index=False); AT.to_csv(OUT/'multihorizon_window_auc.csv', index=False)
DECL = {'status': 'DECLARED before TEST-3 / TEST-B: multi-horizon roll-up of the fusion v0 score (maximum per tumbling window); thresholds and 0-100 scale come from genuine DEV out-of-fold windows.', 'declared_utc': datetime.datetime.utcnow().isoformat(timespec='seconds') + 'Z',
        'score': 'fusion_v0 (declared_fusion_v0.json)', 'horizons_bars': HOR, 'tier_share_of_normal_windows': FRAC, 'thresholds': {s: {h: {t: THR[(s, h, t)] for t in ['Watch', 'Alert']} for h in HOR} for s in SYMBOLS},
        'score100_percentile_points': {s: {h: [float(x) for x in ECDF[(s, h)]] for h in HOR} for s in SYMBOLS}, 'hypotheses': {'H_M1': HM1, 'H_M2': HM2, 'H_M3': HM3}, 'fusion_declared_sha16': sha_file(OUT/'declared_fusion_v0.json'), 'watch_declared_sha16': sha_file(OUT/'declared_watch_v1.json')}
(OUT/'declared_multihorizon_v1.json').write_text(json.dumps(DECL, indent=1))
block = '\n'.join(LINES); (OUT/'b2_step13_share_block.txt').write_text(block)
print('\n===== SHARE BLOCK (attach models_b2_v1/b2_step13_share_block.txt, or copy everything below) ====='); print(block); print('saved', OUT/'b2_step13_share_block.txt')


===== SHARE BLOCK (attach models_b2_v1/b2_step13_share_block.txt, or copy everything below) =====
protocol B2 intact, sha256 443f99d5faf5929f; library sha256 1b0b46630763c944
windows: {'15 min': '2496 genuine DEV, 2880 CAL per symbol', '1 hour': '624 genuine DEV, 720 CAL per symbol', '1 day': '26 genuine DEV, 30 CAL per symbol'}
PASS T01 the 1-minute Alert recomputed here equals notebooks 35 and 40 (macro 8 0.778)
PASS T02 window maximum equals a hand calculation
PASS T03 window-overlap rule equals a plain loop
PASS T04 Watch threshold <= Alert threshold; score100 is monotone and inside 0-100
PASS T05 only CAL-3 and DEV out-of-fold series are read (no TEST key in the score files)

--- detection at window level (an event is found if ANY window overlapping it is flagged) versus the 1-minute Alert (macro 8 0.778) ---
horizon  tier  target genuine share  CAL genuine share flagged  genuine flagged windows/day  macro 8 recall
 15 min Watch                  0.05                      0.041   